# The Nonhierarchical Bayesian Finite Element Method: Pullout Test

This notebook is associated with section 4.1 of "\<paper title\>" by Anne Poot, Marvin Pförtner, Iuri Rocha, Philipp Hennig, Pierre Kerfriden and Frans van der Meer (\<doi link\>).

In [ ]:
# general imports
import os
import numpy as np
import urllib.request
import zipfile

# local imports
from bfem.observation import compute_bfem_observations
from fem.jive import CJiveRunner
from fem.meshing import mesh_interval_with_line2, create_hypermesh
from probability.multivariate import Gaussian
from probability.process import (
    GaussianProcess,
    InverseCovarianceOperator,
    ProjectedPrior,
)
from util.io import read_csv_from
from util.linalg import Matrix

from experiments.reproduction.nonhierarchical.pullout_bar import misc
from experiments.reproduction.nonhierarchical.pullout_bar.props import get_fem_props
from experiments.reproduction.nonhierarchical.pullout_bar.plots import (
    exact_plot,
    bfem_plot,
    scatter_plot,
)

## Forward problem

We first consider the FEM and BFEM solutions to the forward problem.

### Figure 4b: FEM solutions

This figure shows the FEM solution to the pullout test problem for various mesh densities, along with the exact solution.

In [ ]:
n_elems = np.array([1, 2, 4, 8, 16, 32, 64])
us = []

# get fem settings
props = get_fem_props()

# compute fem solution for each mesh size
for n_elem in n_elems:
    nodes, elems = mesh_interval_with_line2(n=n_elem)
    jive = CJiveRunner(props, elems=elems)
    globdat = jive()
    us.append(globdat["state0"])

# plot the results
exact_plot(n_elems, us)

### Figures 5a to 5f: BFEM solutions

The following six figures show the exact BFEM prior, exact BFEM posterior, and the BFEM posterior on 4 reference meshes: refined, dual, left and right.
Further description of the plots can be found in section 4.1.1 of the paper.

In [ ]:
n_elem = 4

combos = [
    ("exact", "prior"),
    ("exact", "posterior"),
    ("hierarchical", "posterior"),
    ("dual", "posterior"),
    ("left", "posterior"),
    ("right", "posterior"),
]

for ref_type, dist_type in combos:

    # generate meshes
    obs_nodes, obs_elems = mesh_interval_with_line2(n=n_elem)

    if ref_type == "exact":
        ref_nodes, ref_elems = mesh_interval_with_line2(n=1024)
    elif ref_type == "hierarchical":
        ref_nodes, ref_elems = mesh_interval_with_line2(n=2 * n_elem)
    elif ref_type == "dual":
        ref_nodes, ref_elems = misc.dual_mesh(obs_elems)
    elif ref_type == "random":
        ref_nodes, ref_elems = misc.random_mesh(n=n_elem, seed=0)
    elif ref_type == "left":
        ref_nodes, ref_elems = mesh_interval_with_line2(n=n_elem + 1)
        ref_nodes._data[: n_elem + 1, 0] = np.linspace(0, 0.5, n_elem + 1)
    elif ref_type == "right":
        ref_nodes, ref_elems = mesh_interval_with_line2(n=n_elem + 1)
        ref_nodes._data[1 : n_elem + 2, 0] = np.linspace(0.5, 1.0, n_elem + 1)
    else:
        assert False

    (hyp_nodes, hyp_elems), hyp_map = create_hypermesh(obs_elems, ref_elems)

    # determine scale by maximizing marginal likelihood
    jive = CJiveRunner(get_fem_props(), elems=obs_elems)
    globdat = jive()
    u_obs = globdat["state0"]
    K_obs = globdat["matrix0"]
    n_obs = len(u_obs)
    alpha2_mle = u_obs @ K_obs @ u_obs / n_obs

    # set up fem solver
    module_props = get_fem_props()
    model_props = module_props.pop("model")
    ref_jive_runner = CJiveRunner(module_props, elems=ref_elems)
    obs_jive_runner = CJiveRunner(module_props, elems=obs_elems)
    hyp_jive_runner = CJiveRunner(module_props, elems=hyp_elems)

    # define prior on exact, reference and observation level
    inf_cov = InverseCovarianceOperator(model_props=model_props, scale=alpha2_mle)
    inf_prior = GaussianProcess(None, inf_cov)
    ref_prior = ProjectedPrior(prior=inf_prior, jive_runner=ref_jive_runner)
    obs_prior = ProjectedPrior(prior=inf_prior, jive_runner=obs_jive_runner)
    hyp_prior = ProjectedPrior(prior=inf_prior, jive_runner=hyp_jive_runner)

    # condition on observation shape functions
    H_obs, f_obs = compute_bfem_observations(obs_prior, hyp_prior)
    H_ref, f_ref = compute_bfem_observations(ref_prior, hyp_prior)

    Phi_obs = H_obs[0].T
    Phi_ref = H_ref[0].T
    K_hyp = H_obs[1]

    K_obs = Matrix((Phi_obs.T @ K_hyp @ Phi_obs).evaluate(), name="K_obs")
    K_ref = Matrix((Phi_ref.T @ K_hyp @ Phi_ref).evaluate(), name="K_ref")
    K_x = Matrix((Phi_ref.T @ K_hyp @ Phi_obs).evaluate(), name="K_x")

    P_obs = Phi_obs @ K_obs.inv @ Phi_obs.T @ K_hyp
    P_ref = Phi_ref @ K_ref.inv @ Phi_ref.T @ K_hyp

    mean = Phi_obs @ obs_prior.globdat["state0"]
    cov = K_ref.inv.evaluate()
    cov -= (K_ref.inv @ K_x @ K_obs.inv @ K_x.T @ K_ref.inv).evaluate()
    cov *= alpha2_mle
    cov = Phi_ref @ (Phi_ref @ cov).T

    prior = Gaussian(None, alpha2_mle * K_ref.inv.evaluate()) @ Phi_ref.T
    posterior = Gaussian(mean, cov, allow_singular=True)

    # plot the results
    if dist_type == "prior":
        dist = prior
    elif dist_type == "posterior":
        dist = posterior
    else:
        assert False

    bfem_plot(dist, globdat=hyp_prior.globdat, dist_type=dist_type)

## Inverse Problem

We now consider the inverse problem, described in section 3.1.1.
The dataset can be regenerated, but it it easier to download it directly.

In [ ]:
cwd = os.getcwd()
tmp_path = os.path.join(cwd, "tmp")
zip_path = os.path.join(tmp_path, "pullout-bar.zip")
output_path = os.path.join(tmp_path, "output")
url = "https://surfdrive.surf.nl/public.php/dav/files/wpZag5RP2WpN6CQ"

if not os.path.exists(tmp_path):
    # download zip file
    os.mkdir(tmp_path)
    out = urllib.request.urlretrieve(url, zip_path)

    # extract in tmp folder
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(tmp_path)

assert os.path.isfile(zip_path)
assert os.path.isdir(output_path)

### Figure 7: inverse problem solutions

These figures show samples from the FEM posterior, and 5 the BFEM posterior on 5 different reference meshes: exact, refined, dual, left and right, as defined in figure 4a of the paper.
Further description of the plots can be found in section 4.1.3 of the paper.

In [ ]:
# rng seed
# seed = 0  # single run
seed = "0-20"  # meta run

if isinstance(seed, int):
    # for the standard run, a burn-in period is needed during which the proposal is adapted
    N_burn = 10000
    N_filter = 50
elif isinstance(seed, str):
    # for the meta run, the proposal is never adapted, so no burn-in period is needed
    N_burn = 25
    N_filter = 50
else:
    assert False

for fem_type in [
    "fem",
    "bfem-exact",
    "bfem-hierarchical",
    "bfem-dual",
    "bfem-left",
    "bfem-right",
]:
    if fem_type in ["fem", "bfem-left"]:
        n_elem_range = [1, 2, 4, 8, 16, 32, 64]
    else:
        n_elem_range = [1, 4, 16, 64]

    # load data, discard burn-in and thin samples
    fname = os.path.join(output_path, "samples-{}_seed-{}.csv".format(fem_type, seed))
    df = read_csv_from(fname, "log_E,log_k")
    df = df[(df["sample"] >= N_burn) & (df["sample"] % N_filter == 0)]
    df = df[df["n_elem"].isin(n_elem_range)]

    # plot the posteriors
    scatter_plot(df)